# 测试集生成

策划高质量的测试数据集对于评估 AI 应用程序的性能至关重要。

- 理想测试数据集的特征
- 包含高质量数据样本
- 涵盖现实世界中观察到的各种场景。
- 包含足够数量的样本以得出具有统计意义的结论。
- 持续更新以防止数据漂移
- 手动整理这样的数据集可能既耗时又昂贵。

Ragas 提供了一套工具来生成用于评估 AI 应用程序的合成测试数据集。


### RAG 的测试集生成
在 RAG 应用程序中，当用户通过应用程序与一组文档进行交互时，系统可能会遇到不同的查询模式。首先，让我们了解一下 RAG 应用程序中可能遇到的不同类型的查询。

RAG 中的查询类型

![](https://cdn.mathpix.com/snip/images/G_VdTlOpitwk0d6PB-kj-KlHf8VEBo7ERfIPDiGd7lU.original.fullsize.png)

#### 单跳查询
单跳查询是一个简单的问题，需要从单个文档或来源检索信息以提供相关答案。它只需一步即可得出答案。

- 示例（具体查询）：

“阿尔伯特·爱因斯坦在哪一年发表了相对论？”

这是一个具体的、基于事实的问题，只需从包含该信息的文档中检索一次即可回答。

- 示例（抽象查询）：

“爱因斯坦的理论如何改变我们对时间和空间的理解？”

虽然这个查询仍然涉及单一概念（相对论），但它需要来自源材料的更抽象或解释性的解释。

#### 多跳查询
多跳查询涉及多个推理步骤，需要来自两个或多个来源的信息。系统必须从各种文档中检索信息，并将这些信息串联起来，才能生成准确的答案。

- 示例（具体查询）：

“哪位科学家影响了爱因斯坦的相对论研究？他们提出了什么理论？”

这要求系统检索有关影响爱因斯坦的科学家和具体理论的信息，可能来自两个不同的来源。

- 示例（抽象查询）：

“自爱因斯坦首次发表相对论以来，科学理论是如何演变的？”

这个抽象的查询需要随着时间的推移和跨不同来源检索多条信息，以形成关于理论演变的广泛的解释性反应。

#### RAG 中的特定查询与抽象查询
具体查询：注重清晰、基于事实的检索。RAG 的目标是从一个或多个直接针对特定问题的文档中检索高度相关的信息。

抽象查询：需要更广泛、更具解释性的响应。在 RAG 中，抽象查询要求检索系统从包含更高层次推理、解释或观点（而非简单事实）的文档中提取信息。

在单跳和多跳情况下，具体查询和抽象查询之间的区别通过确定重点是精确度（具体）还是综合更广泛的想法（抽象）来塑造检索和生成过程。

不同类型的查询需要合成不同的上下文。为了解决这个问题，Ragas 使用基于知识图谱的方法进行测试集生成。



### 知识图谱创建
由于我们希望从给定的文档集中生成不同类型的查询，我们面临的主要挑战是确定正确的块或文档集，以便 LLM 创建查询。为了解决这个问题，Ragas 使用基于知识图谱的方法进行测试集生成。

<img src="https://docs.ragas.io/en/stable/_static/imgs/kg_rag.png">

### 知识图谱创建
知识图谱由以下组件创建：

文档分割器
文档会被分块，形成层级节点。分块可以使用不同的拆分器来完成。例如，对于财务文档，可以使用拆分器根据损益表、资产负债表、现金流量表等部分对文档进行拆分。您可以编写自定义拆分器，根据与您的领域相关的部分对文档进行拆分。


In [ ]:
from ragas.testset.graph import Node

sample_nodes = [Node(
    properties={"page_content": "Einstein's theory of relativity revolutionized our understanding of space and time. It introduced the concept that time is not absolute but can change depending on the observer's frame of reference."}
),Node(
    properties={"page_content": "Time dilation occurs when an object moves close to the speed of light, causing time to pass slower relative to a stationary observer. This phenomenon is a key prediction of Einstein's special theory of relativity."}
)]
sample_nodes

[Node(id: 4f6b94, type: , properties: ['page_content']),            
 Node(id: 952361, type: , properties: ['page_content'])]

![](https://cdn.mathpix.com/snip/images/pSmt3lCBSBYeFAC94ESJxzDiaj0PBZizgFGDYlTBNrg.original.fullsize.png)

#### 提取器
不同的提取器用于从每个节点提取信息，这些信息可用于建立节点之间的关系。例如，在财务文档中，可以使用实体提取器来提取公司名称等实体，使用关键词提取器来提取每个节点中存在的重要关键词等等。您可以编写自定义提取器来提取与您的领域相关的信息。

提取器可以是基于 LLM 的（继承自）LLMBasedExtractor，也可以是基于规则的（继承自）Extractor。

例子

假设我们有一个来自知识图谱的示例节点。我们可以使用NERExtractor从该节点中提取命名实体。

In [ ]:
from ragas.testset.transforms.extractors import NERExtractor

extractor = NERExtractor()
output = [await extractor.extract(node) for node in sample_nodes]
output[0]

返回提取器的类型和提取的信息的元组。

```shell
('entities',
 {'ORG': [],
  'LOC': [],
  'PER': ['Einstein'],
  'MISC': ['theory of relativity',
   'space',
   'time',
   "observer's frame of reference"]})

```



我们将提取的信息添加到节点。

In [ ]:
_ = [node.properties.update({key:val}) for (key,val), node in zip(output, sample_nodes)]
sample_nodes[0].properties

输出：

```json
{'page_content': "Einstein's theory of relativity revolutionized our understanding of space and time. It introduced the concept that time is not absolute but can change depending on the observer's frame of reference.",
 'entities': {'ORG': [],
  'LOC': [],
  'PER': ['Einstein'],
  'MISC': ['theory of relativity',
   'space',
   'time',
   "observer's frame of reference"]}}
```


![](https://cdn.mathpix.com/snip/images/SF_ix2-_NB_M4QVdMz4Ahd5fIEs5lkQqE8GuUnGMh_s.original.fullsize.png)

#### 关系建立者
提取的信息用于建立节点之间的关系。例如，对于财务文档，可以根据节点中存在的实体建立节点之间的关系。您可以编写自己的自定义关系构建器，根据与您的领域相关的信息建立节点之间的关系。

例子



In [1]:
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.transforms.relationship_builders.traditional import JaccardSimilarityBuilder

kg = KnowledgeGraph(nodes=sample_nodes)
rel_builder = JaccardSimilarityBuilder(property_name="entities", key_name="PER", new_property_name="entity_jaccard_similarity")
relationships = await rel_builder.transform(kg)
relationships

/opt/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'sample_nodes' is not defined

输出：

[Relationship(Node(id: 4f6b94) <-> Node(id: 952361), type: jaccard_similarity, properties: ['entity_jaccard_similarity'])]

由于两个节点具有相同的实体“爱因斯坦”，因此节点之间的关系是基于实体相似性建立的。



![](https://cdn.mathpix.com/snip/images/vckZtTdAgSzT9JekGUGWOIVJROgoAySHZA81ZOj1cas.original.fullsize.png)

现在让我们了解如何使用上述组件构建知识图谱transform，这将使您的工作更轻松。

变换


所有用于构建知识图谱的组件都可以组合成一个组件，transform并应用于知识图谱构建。变换由一系列按顺序应用于知识图谱的组件组成。它还可以处理组件的并行处理。该apply_transforms方法用于将变换应用于知识图谱。

让我们使用上述组件来构建上述知识图谱transform。

In [ ]:
from ragas.testset.transforms import apply_transforms
transforms = [
    extractor,
    rel_builder
    ]

apply_transforms(kg,transforms)

为了并行应用几个组件，您可以将它们包装在Parallel类中。

In [ ]:
from ragas.testset.transforms import KeyphraseExtractor, NERExtractor
from ragas.testset.transforms import apply_transforms, Parallel

tranforms = [
    Parallel(
        KeyphraseExtractor(),
        NERExtractor()
    ),
    rel_builder
]

apply_transforms(kg,transforms)

创建知识图谱后，可以通过遍历图谱生成不同类型的查询。例如，要生成查询“比较 X 公司和 Y 公司从 2020 财年到 2023 财年的收入增长情况”，可以遍历图谱，找到包含 X 公司和 Y 公司从 2020 财年到 2023 财年收入增长信息的节点。

#### 场景生成
现在，我们拥有了知识图谱，可以用来构建合适的上下文来生成任何类型的查询。当用户群体与 RAG 系统交互时，他们可能会根据自己的角色（例如，高级工程师、初级工程师等）、查询长度（短、长等）、查询风格（正式、非正式等）以各种方式构建查询。为了生成涵盖所有这些场景的查询，Ragas 使用基于场景的方法进行测试集生成。

测试集生成中的每个参数Scenario都是以下参数的组合。

- 节点：用于生成查询的节点
- 查询长度：所需查询的长度，可以是短、中、长等。
- 查询风格：查询的风格，可以是网页搜索、聊天等。
- 角色：用户的角色，可以是高级工程师、初级工程师等。

（即将推出）

<img src="https://docs.ragas.io/en/stable/_static/imgs/scenario_rag.png">

#### 查询合成器
负责QuerySynthesizer为单个查询类型生成不同的场景。generate_scenarios方法用于为单个查询类型生成场景。generate_sample方法用于为单个场景生成查询和参考答案。让我们通过一个例子来理解这一点。

例子

在前面的例子中，我们创建了一个知识图谱，其中包含两个基于实体相似性相互关联的节点。现在假设你的知识图谱中有 20 对这样的节点，它们基于实体相似性相互关联。

假设你的目标是创建 50 个不同的查询，每个查询都是关于比较两个实体的抽象问题。首先，我们必须查询知识图谱 (KG)，根据实体相似度获取彼此相关的节点对。然后，我们必须为每对节点生成场景，直到获得 50 个不同的场景。此逻辑在generate_scenarios方法中实现。

In [ ]:
from dataclasses import dataclass
from ragas.testset.synthesizers.base_query import QuerySynthesizer

@dataclass
class EntityQuerySynthesizer(QuerySynthesizer):

    async def _generate_scenarios( self, n, knowledge_graph, callbacks):
        """
        logic to query nodes with entity
        logic describing how to combine nodes,styles,length,persona to form n scenarios
        """

        return scenarios

    async def _generate_sample(
        self, scenario, callbacks
    ):

        """
        logic on how to use tranform each scenario to EvalSample (Query,Context,Reference)
        you may create singleturn or multiturn sample
        """

        return SingleTurnSample(user_input=query, reference_contexs=contexts, reference=reference)

### 代理或工具用例的测试集生成

评估代理或工具使用工作流程可能颇具挑战性，因为它涉及多个步骤和交互。策划一个涵盖所有可能场景和边缘情况的测试套件尤其困难。我们正在开发一套工具，用于生成用于评估代理工作流程的合成测试数据。